**Archived large-GPU configuration. For the free T4, use LocalCodeGraph_Gemma_Repair.ipynb.**

# LocalCodeGraph: paired Gemma repair pilot

This notebook generates one patch per task for BM25 and graph context on all 23 SWE-bench Lite dev issues. It does **not** grade repairs or run the official ADK competition agent. No Gemma performance result exists until these predictions are generated and independently tested.

Select **Runtime → Change runtime type → an A100 GPU** (or a runtime with at least 38,000 MiB GPU memory). GPU types and availability are not guaranteed by Colab; a free T4 is too small for this notebook's chosen 31B configuration. The GPU check runs before large downloads. Do not purchase compute automatically on behalf of this notebook.

Run cells in order. Model dependencies are isolated in a virtual environment. Initial setup requires internet; inference uses a loopback-only local endpoint. Keep the runtime active through ordinary notebook use; no keep-alive workarounds are included.

[Protocol](https://github.com/eluan216/LocalCodeGraph/blob/main/benchmarks/REPAIR_PROTOCOL.md) · [Colab resource limits](https://research.google.com/colaboratory/faq.html) · [Model](https://huggingface.co/google/gemma-4-31B-it-qat-w4a16-ct)


In [ ]:
import json, os, pathlib, shutil, subprocess, sys, time, urllib.request
ROOT = pathlib.Path("/content/LocalCodeGraph")
VENV = pathlib.Path("/content/lcg-gemma-env")
MODEL_DIR = pathlib.Path("/content/gemma-4-31b-qat")
INPUTS = pathlib.Path("/content/lcg-repair-inputs")
OUTPUT = pathlib.Path("/content/lcg-repair-run")
VLLM_VERSION = "0.30.0"
MODEL = "google/gemma-4-31B-it-qat-w4a16-ct"

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True)
print(gpu.stdout)
if not any(int(line.rsplit(",", 1)[1]) >= 38000 for line in gpu.stdout.strip().splitlines()):
    raise RuntimeError("This 31B pilot needs a GPU with at least 38,000 MiB. Select an A100-class runtime.")
if shutil.disk_usage("/content").free < 65 * 1024**3:
    raise RuntimeError("At least 65 GiB free disk is required before installing/downloading.")
if INPUTS.exists() or OUTPUT.exists():
    raise RuntimeError("Existing inputs/output found. Preserve them; use a fresh runtime for a new pilot.")


## Install in an isolated environment

This cell may take several minutes. It pins vLLM for this pilot and records the final environment. CPU CI validates notebook structure and controller tests; the GPU setup still needs validation on your Colab runtime.


In [ ]:
if not ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/eluan216/LocalCodeGraph.git", str(ROOT)], check=True)
REPO_COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
print("Toolkit commit:", REPO_COMMIT)
subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
PYTHON = str(VENV / "bin/python")
VLLM = str(VENV / "bin/vllm")
subprocess.run([PYTHON, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "-e", str(ROOT),
                "vllm==" + VLLM_VERSION, "pyarrow==26.0.1", "unidiff==1.0.1",
                "huggingface_hub"], check=True)


## Build the gold-free paired inputs

Only issue descriptions and pre-fix source context are passed to Gemma. Reference/test patches and expected outcomes are omitted. Source snippets retain upstream licenses, copied into the input bundle. This is a public development set already used for retrieval analysis.


In [ ]:
subprocess.run([PYTHON, str(ROOT / "benchmarks/prepare_repair_inputs.py"),
                "--output-dir", str(INPUTS)], check=True)
manifest = json.loads((INPUTS / "manifest.json").read_text())
print("Tasks:", manifest["tasks"], "Input SHA256:", manifest["inputs_sha256"])
assert manifest["tasks"] == 23


## Download the exact model snapshot

The public model may download without authentication. If access is requested, accept its terms on Hugging Face and put a token in Colab Secrets as `HF_TOKEN`; never paste a token into a cell or commit it. The token is not written to result files. An immutable model revision is resolved before downloading and recorded for the run.


In [ ]:
download_env = os.environ.copy()
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        download_env["HF_TOKEN"] = token
except Exception:
    pass
model_code = r'''
import json, sys
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
model, destination = sys.argv[1:]
revision = HfApi().model_info(model).sha
snapshot_download(model, revision=revision, local_dir=destination)
Path(destination, "lcg-revision.json").write_text(json.dumps({"model": model, "revision": revision}))
print("Model revision:", revision)
'''
subprocess.run([PYTHON, "-c", model_code, MODEL, str(MODEL_DIR)],
               env=download_env, check=True)
MODEL_REVISION = json.loads((MODEL_DIR / "lcg-revision.json").read_text())["revision"]
download_env.pop("HF_TOKEN", None)


## Start the local inference server

No public tunnel or external API is used. If loading fails or runs out of memory, inspect the log; do not substitute a smaller model and report it as this pilot.


In [ ]:
SERVER_LOG = pathlib.Path("/content/lcg-vllm.log")
server_env = os.environ.copy()
server_env.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
server_log = SERVER_LOG.open("w")
server = subprocess.Popen([
    VLLM, "serve", str(MODEL_DIR), "--served-model-name", MODEL,
    "--host", "127.0.0.1", "--port", "8000",
    "--tensor-parallel-size", "1", "--max-model-len", "32768",
    "--max-num-seqs", "1", "--gpu-memory-utilization", "0.90",
    "--generation-config", "vllm",
], stdout=server_log, stderr=subprocess.STDOUT, env=server_env)
deadline = time.monotonic() + 1200
while time.monotonic() < deadline:
    if server.poll() is not None:
        raise RuntimeError("vLLM exited. Log tail:\n" + SERVER_LOG.read_text()[-6000:])
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=3) as response:
            advertised = json.load(response)
        if MODEL in {entry["id"] for entry in advertised["data"]}:
            print("Local Gemma server ready.")
            break
    except Exception:
        pass
    time.sleep(10)
else:
    raise TimeoutError("Server startup timed out; inspect /content/lcg-vllm.log.")


## Generate both arms

This makes 46 requests (23 per arm), with fixed generation parameters. Raw responses, errors, token usage and patches are saved. Weight hashing records the actual local files and can take several minutes before the first request. Do not rerun completed attempts to select better patches.


In [ ]:
subprocess.run([PYTHON, "-m", "localcodegraph.repair",
    "--inputs", str(INPUTS / "inputs.jsonl"),
    "--manifest", str(INPUTS / "manifest.json"),
    "--model-dir", str(MODEL_DIR), "--model-revision", MODEL_REVISION,
    "--output-dir", str(OUTPUT)], check=True)
environment = subprocess.check_output([PYTHON, "-m", "pip", "freeze"], text=True)
(OUTPUT / "environment.txt").write_text(environment)
(OUTPUT / "hardware.txt").write_text(subprocess.check_output(["nvidia-smi"], text=True))
(OUTPUT / "toolkit-commit.txt").write_text(REPO_COMMIT + "\n")
shutil.copyfile(SERVER_LOG, OUTPUT / "vllm.log")
traces = [json.loads(line) for line in (OUTPUT / "traces.jsonl").read_text().splitlines()]
from collections import Counter
print(Counter((row["arm"], row["status"]) for row in traces))
print("These are generation statuses, not resolved-task scores.")


## Download outputs

Save this ZIP before Colab disconnects. It includes prediction JSONL files for the official grader. It excludes model weights. Keep incomplete attempts and error traces.


In [ ]:
shutil.copytree(INPUTS, OUTPUT / "gold-free-inputs")
archive = shutil.make_archive("/content/localcodegraph-gemma-repair", "zip", OUTPUT)
from google.colab import files
files.download(archive)


## Stop the server

Docker-based SWE-bench verification is a separate step on a compatible host. See `colab/README.md` for the official commands, split selection and control requirements. Do not label generated patches as successful repairs before grading.


In [ ]:
if server.poll() is None:
    server.terminate()
    try:
        server.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()
server_log.close()
print("Local inference server stopped.")
